# StandUp4AI Evaluation: F0 Prosody vs Baseline

**Goal:** Evaluate if our F0/spectral features beat StandUp4AI's F1=0.51 on their benchmark.

**Data:** ~30 videos (we have audio+labels for these).

**Baseline to beat:** F1=0.51 (StandUp4AI text-based)

In [ ]:
# Setup
import os, sys, json
from pathlib import Path
import subprocess

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Try different path formats
possible_paths = [
    Path('/content/drive/MyDrive/standup4ai'),
    Path('/content/drive/Shareddrives/standup4ai'),
    Path('/content/drive/MyDrive/standup4ai/audio'),
]

BASE = None
for p in possible_paths:
    if p.exists():
        BASE = p
        print(f'Found at: {BASE}')
        break

if BASE is None:
    # List what exists
    print('Checking paths...')
    for p in possible_paths:
        print(f'  {p}: exists={p.exists()}')
    # Try listing
    result = subprocess.run(['ls', '/content/drive/MyDrive/'], capture_output=True, text=True, timeout=10)
    print(f'/content/drive/MyDrive/ contents:\n{result.stdout[:500]}')
    
    # Check if maybe different folder name
    result = subprocess.run(['ls', '/content/drive/'], capture_output=True, text=True, timeout=10)
    print(f'/content/drive/ contents:\n{result.stdout[:500]}')
    
    raise FileNotFoundError('standup4ai folder not found in Drive')

AUDIO_DIR = BASE / 'audio'
LABELS_DIR = BASE / 'labels'

# List what we have
audio_files = list(AUDIO_DIR.glob('*.m4a')) + list(AUDIO_DIR.glob('*.mp3'))
label_files = list(LABELS_DIR.glob('*.csv'))
print(f'Audio files: {len(audio_files)}')
print(f'Label files: {len(label_files)}')

# Get overlapping video IDs
audio_vids = {f.stem for f in audio_files}
label_vids = {f.stem for f in label_files}
overlap = audio_vids & label_vids
print(f'Have both audio+labels: {len(overlap)} videos')

if len(overlap) == 0:
    print('\nWARNING: No overlap! Checking formats...')
    print(f'Audio vids sample: {sorted(audio_vids)[:3]}')
    print(f'Label vids sample: {sorted(label_vids)[:3]}')

# Install deps
subprocess.run(['pip', 'install', '-q', 'librosa', 'pandas', 'numpy', 'scikit-learn'], check=True, timeout=60)
import librosa, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load labels for overlapping videos
def load_laughter_labels(csv_path):
    """Load laughter intervals from CSV."""
    df = pd.read_csv(csv_path)
    # Handle different column names
    if 'label' not in df.columns:
        print(f'WARNING: No label column in {csv_path.name}. Columns: {list(df.columns)}')
        return None
    # Binary: risa=1, no_risa=0
    df['label_bin'] = (df['label'].str.strip() == 'risa').astype(int)
    return df

# Test with one file
sample_vid = sorted(overlap)[0]
sample_audio = AUDIO_DIR / f'{sample_vid}.m4a'
sample_labels = LABELS_DIR / f'{sample_vid}.csv'

print(f'Testing with: {sample_vid}')
print(f'  Audio exists: {sample_audio.exists()}')
print(f'  Labels exists: {sample_labels.exists()}')

if sample_labels.exists():
    labels_df = load_laughter_labels(sample_labels)
    if labels_df is not None:
        print(f'  Labels shape: {labels_df.shape}')
        print(f'  Label values: {labels_df["label"].value_counts().to_dict()}')
        print(f'  Positive rate: {labels_df["label_bin"].mean():.1%}')

In [ ]:
# Extract prosody features for a segment
def extract_features(audio_path, t0, t1, sr=22050):
    try:
        dur = min(float(t1) - float(t0), 10.0)
        if dur < 0.1:
            return None
        
        # Load audio segment
        y, sr = librosa.load(str(audio_path), sr=sr, offset=float(t0), duration=dur, mono=True)
        
        if len(y) < sr * 0.1:
            return None
        
        # F0 via pyin (fundamental frequency)
        try:
            f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=80, fmax=500, sr=sr, hop_length=512)
            f0 = np.nan_to_num(f0, nan=0)
            f0_mean = float(np.mean(f0))
            f0_std = float(np.std(f0))
            voiced_rate = float(np.mean(voiced_flag))
        except:
            f0_mean, f0_std, voiced_rate = 0.0, 0.0, 0.0
        
        # Spectral features
        hop = 512
        
        # RMS energy
        rms = librosa.feature.rms(y=y, hop_length=hop)[0]
        
        # Zero crossing rate
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        
        # Spectral features
        cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        flat = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        
        # MFCCs
        mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop, n_mfcc=13)
        
        # Duration
        duration = len(y) / sr
        
        return np.array([
            f0_mean, f0_std, voiced_rate,
            float(np.mean(rms)), float(np.std(rms)), float(np.max(rms)),
            float(np.mean(zcr)), float(np.std(zcr)),
            float(np.mean(cent)), float(np.std(cent)),
            float(np.mean(bw)), float(np.std(bw)),
            float(np.mean(flat)), float(np.std(flat)),
            float(np.mean(mfcc[1])), float(np.std(mfcc[1])),
            float(np.mean(mfcc[2])), float(np.std(mfcc[2])),
            float(np.mean(mfcc[3])), float(np.std(mfcc[3])),
            duration
        ], dtype=np.float32)
        
    except Exception as e:
        return None

# Test extraction on one segment
if sample_labels.exists():
    labels_df = load_laughter_labels(sample_labels)
    if labels_df is not None and len(labels_df) > 0:
        seg = labels_df.iloc[0]
        feat = extract_features(sample_audio, seg['t0'], seg['t1'])
        if feat is not None:
            print(f'Feature test OK: shape={feat.shape}, values={feat[:3]}')
        else:
            print('Feature test FAILED')

In [ ]:
# Extract features for ALL overlapping videos
print(f'\nExtracting features from {len(overlap)} videos...')

X_all, y_all, vids_all = [], [], []
failed = []

for i, vid in enumerate(sorted(overlap)):
    audio_path = AUDIO_DIR / f'{vid}.m4a'
    label_path = LABELS_DIR / f'{vid}.csv'
    
    if not audio_path.exists():
        failed.append((vid, 'audio not found'))
        continue
    if not label_path.exists():
        failed.append((vid, 'labels not found'))
        continue
    
    labels_df = load_laughter_labels(label_path)
    if labels_df is None:
        failed.append((vid, 'labels failed to load'))
        continue
    
    for _, seg in labels_df.iterrows():
        feat = extract_features(audio_path, seg['t0'], seg['t1'])
        if feat is not None:
            X_all.append(feat)
            y_all.append(int(seg['label_bin']))
            vids_all.append(vid)
    
    if (i + 1) % 5 == 0:
        print(f'  Processed {i+1}/{len(overlap)} videos, {len(X_all)} samples so far...')

X = np.array(X_all) if X_all else np.array([])
y = np.array(y_all)
videos = np.array(vids_all)

print(f'\n=== DATASET ===')
print(f'Samples: {len(y)}')
print(f'Positive: {y.sum()} ({y.mean():.1%})')
print(f'Videos: {len(set(videos))}')
if len(X) > 0:
    print(f'Features: {X.shape[1]}-dim')

if failed:
    print(f'\nFailed: {len(failed)} videos')
    for vid, reason in failed[:3]:
        print(f'  {vid}: {reason}')

In [ ]:
# Skip if no data
if len(y) < 10:
    print(f'ERROR: Only {len(y)} samples. Cannot train.')
    print('\nDebugging info:')
    print(f'Overlap: {sorted(overlap)[:10]}')
    # Try manual check
    test_vid = sorted(overlap)[0]
    print(f'\nTrying to manually extract from {test_vid}...')
    test_audio = AUDIO_DIR / f'{test_vid}.m4a'
    test_labels = LABELS_DIR / f'{test_vid}.csv'
    print(f'  Audio: {test_audio} exists={test_audio.exists()}')
    print(f'  Labels: {test_labels} exists={test_labels.exists()}')
    
    if test_labels.exists():
        df = load_laughter_labels(test_labels)
        print(f'  Labels df: {df.shape if df is not None else None}')
        if df is not None and len(df) > 0:
            print(f'  First row: t0={df.iloc[0]["t0"]}, t1={df.iloc[0]["t1"]}, label={df.iloc[0]["label"]}')
            print(f'  Testing feature extraction...')
            feat = extract_features(test_audio, df.iloc[0]['t0'], df.iloc[0]['t1'])
            print(f'  Result: {feat is not None}')
            if feat is not None:
                print(f'  Shape: {feat.shape}')
    
    raise ValueError('Not enough training data')

# Video-level cross-validation
print('\n=== VIDEO-LEVEL CV ===')

models = {
    'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1),
    'XGBoost': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
}

n_splits = min(5, len(set(videos)))
if n_splits < 2:
    print(f'WARNING: Only {n_splits} unique videos. Using leave-one-out.')
    n_splits = len(set(videos))

gkf = GroupKFold(n_splits=n_splits)
results = {}

for name, model in models.items():
    f1s, precs, recs = [], [], []
    for tr, te in gkf.split(X, y, videos):
        if len(set(y[te])) < 2:
            continue
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        model.fit(Xtr, y[tr])
        pred = model.predict(Xte)
        f1s.append(f1_score(y[te], pred, zero_division=0))
        precs.append(precision_score(y[te], pred, zero_division=0))
        recs.append(recall_score(y[te], pred, zero_division=0))
    
    mean_f1 = np.mean(f1s) if f1s else 0
    std_f1 = np.std(f1s) if f1s else 0
    results[name] = {'f1': mean_f1, 'std': std_f1, 'folds': f1s}
    print(f'{name:12s} F1={mean_f1:.4f} ± {std_f1:.4f}')

print(f'\n{"="*50}')
print(f'StandUp4AI baseline: F1=0.51')
for name, r in sorted(results.items(), key=lambda x: -x[1]['f1']):
    beat = '🏆 BEATS BASELINE' if r['f1'] > 0.51 else ''
    print(f'{name:12s} F1={r["f1"]:.4f} ± {r["std"]:.4f} {beat}')
print(f'{"="*50}')

In [ ]:
# Feature importance
print('\n=== FEATURE IMPORTANCE ===')

sc = StandardScaler()
Xs = sc.fit_transform(X)
lr = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1)
lr.fit(Xs, y)

feature_names = ['f0_mean', 'f0_std', 'voiced_rate',
                'rms_mean', 'rms_std', 'rms_max',
                'zcr_mean', 'zcr_std',
                'cent_mean', 'cent_std',
                'bw_mean', 'bw_std',
                'flat_mean', 'flat_std',
                'mfcc2_mean', 'mfcc2_std',
                'mfcc3_mean', 'mfcc3_std',
                'mfcc4_mean', 'mfcc4_std',
                'duration']

coefs = np.abs(lr.coef_[0])
imp = sorted(zip(feature_names, coefs), key=lambda x: -x[1])[:10]
print('Top features:')
for name, coef in imp:
    print(f'  {name:15s} {coef:.4f}')

# Save results
final_results = {
    'dataset': 'StandUp4AI_subset',
    'n_samples': int(len(y)),
    'n_videos': int(len(set(videos))),
    'positive_rate': float(y.mean()),
    'features': f'{X.shape[1]}-dim (F0 + spectral)',
    'models': {k: {'f1': float(v['f1']), 'std': float(v['std'])} for k, v in results.items()},
    'standup4ai_baseline': 0.51,
    'top_features': [(n, float(c)) for n, c in imp[:5]],
}

result_path = BASE / 'standup4ai_results.json'
with open(result_path, 'w') as f:
    json.dump(final_results, f, indent=2)

print(f'\n✅ Results saved to {result_path}')